# Week 3 · Tutorial 3 — Many variables at once, and colour

**Covers videos 3.6 – 3.7:** scatter matrices and projections (PCA) · colour, text and legends.

**Dataset:** `household_profiles.csv` — 800 households summarised by seven numbers and one category each. **One row is one household.**

| column | meaning |
|---|---|
| `mean_load_kw`, `peak_load_kw` | average and highest load |
| `evening_share`, `night_share` | share of energy used 17 – 21 h and 0 – 5 h |
| `winter_summer_ratio` | winter consumption divided by summer consumption |
| `weekend_ratio` | weekend consumption divided by weekday consumption |
| `pv_export_kwh` | yearly energy exported from rooftop PV (0 if none) |
| `heating_type` | `heat pump`, `direct electric`, `district heat`, `wood stove` |

PCA is used here only as a **projection to look at the data** (no prediction involved): it rotates the axes so that the first axes
carry as much of the variation as possible.

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
hp = pd.read_csv("data/household_profiles.csv")
num_cols = [c for c in hp.columns if c != "heating_type"]
print(hp.shape)
hp.head()

## 1. Every pair at once

A scatter matrix draws all pairs of variables. With seven variables that is 21 pairs — fine to look at, but impossible with 80.
The correlation matrix summarises the same pairs with numbers.

In [ ]:
pd.plotting.scatter_matrix(hp[num_cols], figsize=(9, 9), s=6, alpha=0.4, diagonal="hist")
plt.show()

### ✏️ Your turn 1

List the pairs of variables that move together.

The starter lists pairs with an absolute correlation above `thr = 0.9`. Change `thr` to **0.5** and run again. The cell prints the
number of such pairs and the strongest pair with its correlation (2 decimals).

**What to change** — edit the cell exactly like this, then run it:

Change `thr = 0.9` into `thr = 0.5`

In [ ]:
thr = 0.9        # TODO: 0.5

corr = hp[num_cols].corr()
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(num_cols) for b in num_cols[i + 1:] if abs(corr.loc[a, b]) > thr]
pairs.sort(key=lambda t: -abs(t[2]))

print("pairs with |r| >", thr, ":", len(pairs))
for a, b, r in pairs:
    print(f"  {a:20s} {b:20s} {r:6.2f}")
allpairs = sorted(((a, b, corr.loc[a, b]) for i, a in enumerate(num_cols) for b in num_cols[i + 1:]), key=lambda t: -abs(t[2]))
print("strongest pair overall:", allpairs[0][0], "&", allpairs[0][1], round(allpairs[0][2], 2))

## 2. Projections: how much survives?

PCA finds new axes (principal components) ordered by how much of the total variance they carry.
**Scale matters**: PCA works on variances, so a variable measured in large units (`pv_export_kwh`, thousands of kWh) dominates
a variable measured in small ones (`evening_share`, tenths) unless we **standardise** first (mean 0, standard deviation 1).

### ✏️ Your turn 2

Compare PCA with and without standardisation.

Fill in the `TODO`: standardise the data with `StandardScaler` (`X_scaled = StandardScaler().fit_transform(hp[num_cols])`) and run PCA on it. The cell prints the
explained-variance ratios of the first three components for both versions (3 decimals), and the **number of components needed to explain at least 90 %** of the variance (scaled version).

**What to change** — edit the cell exactly like this, then run it:

Change `X_scaled = X_raw.copy()` into `X_scaled = StandardScaler().fit_transform(X_raw)`

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_raw = hp[num_cols].to_numpy()
X_scaled = X_raw.copy()           # TODO: standardise with StandardScaler

pca_raw = PCA().fit(X_raw)
pca_scaled = PCA().fit(X_scaled)

print("raw    :", pca_raw.explained_variance_ratio_[:3].round(3))
print("scaled :", pca_scaled.explained_variance_ratio_[:3].round(3))
cum = np.cumsum(pca_scaled.explained_variance_ratio_)
print("components for >= 90 % variance (scaled):", int(np.argmax(cum >= 0.9) + 1))
print("cumulative variance of the first two components (scaled):", round(cum[1], 3))

### What are the components made of?

### ✏️ Your turn 3

Read the **loadings** of the first two (scaled) components: `pca_scaled.components_[0]` and `[1]` give the weight of each of the
seven variables.

Fill in the `TODO`s: `top_pc1` = the variable with the largest absolute loading on PC 1 and `top_pc2` the same for PC 2; print also the loading of `mean_load_kw` on PC 1 (3 decimals, sign may differ between runs/versions — report the absolute value).

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
top_pc1 = None
top_pc2 = None
```
by
```python
top_pc1 = loadings["PC1"].abs().idxmax()
top_pc2 = loadings["PC2"].abs().idxmax()
```

In [ ]:
loadings = pd.DataFrame(pca_scaled.components_[:2].T, index=num_cols, columns=["PC1", "PC2"])
print(loadings.round(3))

top_pc1 = None      # TODO
top_pc2 = None      # TODO
print("largest |loading| on PC1:", top_pc1)
print("largest |loading| on PC2:", top_pc2)
print("|loading| of mean_load_kw on PC1:", round(abs(loadings.loc["mean_load_kw", "PC1"]), 3))

## 3. Colour it by something you know

If the structure in the projection lines up with a variable you understand, you have learned something. Here we colour the
two-dimensional projection by `heating_type`.

In [ ]:
scores = pca_scaled.transform(X_scaled)[:, :2]
hp["PC1"], hp["PC2"] = scores[:, 0], scores[:, 1]

fig, ax = plt.subplots(figsize=(6, 4.5))
for t, g in hp.groupby("heating_type"):
    ax.scatter(g["PC1"], g["PC2"], s=10, alpha=0.6, label=t)
ax.legend(); ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
plt.show()

### ✏️ Your turn 4

Quantify how well the colour explains each component with the **share of variance explained by the group**
(η² = between-group sum of squares / total sum of squares).

Fill in the `TODO`s: `eta2_pc1` and `eta2_pc2` (3 decimals). For one score column `s` and its group column `g`:
`total = ((s - s.mean())**2).sum()` and `between = sum over groups of n_group · (group mean − overall mean)²`.
Print also the heating type with the **highest mean PC 1 score**.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
    between = np.nan
```
by
```python
    g = values.groupby(groups)
    between = (g.size() * (g.mean() - values.mean()) ** 2).sum()
```

In [ ]:
def eta_squared(values, groups):
    total = ((values - values.mean()) ** 2).sum()
    between = np.nan            # TODO
    return between / total

eta2_pc1 = eta_squared(hp["PC1"], hp["heating_type"])
eta2_pc2 = eta_squared(hp["PC2"], hp["heating_type"])

print("eta^2 PC1:", round(eta2_pc1, 3))
print("eta^2 PC2:", round(eta2_pc2, 3))
print("mean PC1 per heating type:")
print(hp.groupby("heating_type")["PC1"].mean().round(2))

## 4. Colour, text and legends

A colour map is only honest if its **lightness changes steadily**: then equal steps in the data look like equal steps in the picture.
The rainbow (`jet`) has bright and dark sections in the wrong places and creates false “bands”. `viridis` has been built to increase
steadily in lightness (and is readable in greyscale and for most colour-blind readers).

We measure the lightness along the colour map and count how often it **reverses direction**.

In [ ]:
def luminance(rgb):
    # relative luminance (WCAG): linearise the sRGB channels, then weight them
    rgb = np.asarray(rgb, dtype=float)
    lin = np.where(rgb <= 0.03928, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    return 0.2126 * lin[..., 0] + 0.7152 * lin[..., 1] + 0.0722 * lin[..., 2]

def reversals(cmap_name):
    rgb = plt.get_cmap(cmap_name)(np.linspace(0, 1, 256))[:, :3]
    lum = luminance(rgb)
    step = np.sign(np.diff(lum))
    step = step[step != 0]
    return int((np.diff(step) != 0).sum())     # how many times the direction changes

for name in ["viridis", "jet"]:
    print(f"{name:10s} lightness reversals: {reversals(name)}")

### ✏️ Your turn 5

Extend the list of colour maps to `["viridis", "jet", "turbo", "coolwarm", "cividis"]` and run the cell. Report the number of reversals
for each. Which map(s) would you avoid for a sequential quantity like consumption?

**What to change** — edit the cell exactly like this, then run it:

Change `cmaps = ["viridis", "jet"]` into `cmaps = ["viridis", "jet", "turbo", "coolwarm", "cividis"]`

In [ ]:
cmaps = ["viridis", "jet"]      # TODO: ["viridis", "jet", "turbo", "coolwarm", "cividis"]

rev = {name: reversals(name) for name in cmaps}
for name, r in rev.items():
    print(f"{name:10s} lightness reversals: {r}")
print("maps with no reversal:", [n for n, r in rev.items() if r == 0])

### Red and green? Orange and blue?

About 1 in 12 men cannot tell red from green. Lightness *contrast* between two colours helps everybody, and the WCAG
**contrast ratio** `(L_light + 0.05) / (L_dark + 0.05)` measures it (1 = identical lightness, 21 = black on white).

### ✏️ Your turn 6

Complete the function `contrast(c1, c2)`: compute the relative luminance of both colours with `luminance` (given above) and return the contrast ratio
`(L_light + 0.05) / (L_dark + 0.05)`. Then report (2 decimals) the contrast for the two pairs:
- red `#d62728` vs. green `#2ca02c`
- orange `#e69f00` vs. blue `#0072b2`

**What to change** — edit the cell exactly like this, then run it:

Change `return 1.0` into `return (max(L1, L2) + 0.05) / (min(L1, L2) + 0.05)`

In [ ]:
from matplotlib.colors import to_rgb

def contrast(c1, c2):
    L1, L2 = luminance(to_rgb(c1)), luminance(to_rgb(c2))
    return 1.0    # TODO: (L_light + 0.05) / (L_dark + 0.05)

print("red / green   :", round(contrast("#d62728", "#2ca02c"), 2))
print("orange / blue :", round(contrast("#e69f00", "#0072b2"), 2))

## Summary

* A scatter matrix shows every pair; PCA gives a 2-D view of many variables — **standardise first**, then check how much variance survives.
* Colour the projection by something you understand; structure that lines up with a known variable is a finding.
* Use perceptually uniform colour maps (`viridis`), not the rainbow; do not rely on red/green; check lightness contrast.